# LeafRust — полностью автоматическое обучение MobileNetV3 → ONNX/TFLite

Этот ноутбук рассчитан на **Run All** без ручного переключения режимов.

Что автоматизировано:

- поиск LeafRust и автоматическое клонирование репозитория при его отсутствии;
- установка недостающих Python-зависимостей;
- чтение HF-токена из уже поддерживаемых проектом источников;
- скачивание/проверка датасета и весов;
- автоматический выбор CUDA/CPU, AMP, `num_workers` и batch size;
- проверка изображений без удаления исходных файлов;
- стратифицированный train/validation split;
- автоматическое возобновление после прерывания с `last_torch.pt`;
- совместимость со старым `best_torch.pt` как warm-start;
- head-training + fine-tuning, scheduler и early stopping;
- единый глобальный best checkpoint, который не затирается более слабой фазой;
- история обучения, confusion matrix, precision/recall/F1 по классам;
- экспорт ONNX/TFLite, backup предыдущей модели, проверка TFLite inference;
- автоматическое копирование модели в Android assets, если каталог проекта существует;
- итоговый ZIP-пакет и JSON-отчёт.

### Запуск

В Jupyter: **Run → Run All Cells**.

Headless-вариант после сохранения ноутбука в репозитории:

```bash
jupyter nbconvert --to notebook --execute scripts/train_mobilenet_auto.ipynb \
  --output scripts/train_mobilenet_auto.executed.ipynb \
  --ExecutePreprocessor.timeout=-1
```

Параметры при необходимости задаются через переменные окружения, но для обычного запуска ничего менять не требуется.

## 1. Bootstrap: репозиторий и зависимости

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import hashlib
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

REPO_GIT_URL = os.environ.get("LEAFRUST_GIT_URL", "https://github.com/reinethernal/leafrust.git")
DEFAULT_REPO = Path(os.environ.get("LEAFRUST_ROOT", str(Path.home() / "leafrust"))).expanduser().resolve()
FORCE_RECLONE = os.environ.get("LEAFRUST_FORCE_RECLONE", "").strip().lower() in {"1", "true", "yes"}


def _is_leafrust_root(path: Path) -> bool:
    # Достаточно train-скрипта; ensure_train_assets может появиться после обновления
    return (path / "scripts" / "train_mobilenet_torch.py").is_file()


def _candidate_roots() -> list[Path]:
    roots: list[Path] = []
    env = os.environ.get("LEAFRUST_ROOT")
    if env:
        roots.append(Path(env).expanduser().resolve())
    here = Path.cwd().resolve()
    roots.extend([here, *here.parents])
    roots.extend(
        [
            DEFAULT_REPO,
            Path.home() / "leafrust",
            Path.home() / "leafrust-src",
            Path.home() / "CODE" / "leaf",
            Path("/workspace/leafrust"),
            Path("/data/leafrust"),
            Path("/content/leafrust"),
        ]
    )
    seen: set[str] = set()
    out: list[Path] = []
    for r in roots:
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out


def _find_existing_repo() -> Path | None:
    for cand in _candidate_roots():
        try:
            if _is_leafrust_root(cand):
                return cand
        except Exception:
            continue
    return None


def _try_git_update(path: Path) -> bool:
    git = shutil.which("git")
    if not git or not (path / ".git").exists():
        return False
    try:
        subprocess.run([git, "-C", str(path), "fetch", "--depth", "1", "origin"], check=False)
        subprocess.run(
            [git, "-C", str(path), "reset", "--hard", "origin/main"],
            check=False,
            capture_output=True,
        )
        if not _is_leafrust_root(path):
            subprocess.run(
                [git, "-C", str(path), "reset", "--hard", "origin/master"],
                check=False,
                capture_output=True,
            )
        return _is_leafrust_root(path)
    except Exception as exc:
        print("git update failed:", exc)
        return False


def _download_zip_into(target: Path) -> Path:
    if target.exists():
        shutil.rmtree(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    base = REPO_GIT_URL.removesuffix(".git").replace("github.com/", "codeload.github.com/")
    last_error = None
    for branch in ("main", "master"):
        try:
            archive_url = f"{base}/zip/refs/heads/{branch}"
            print("ZIP download:", archive_url)
            with tempfile.TemporaryDirectory(prefix="leafrust-bootstrap-") as td:
                td_path = Path(td)
                archive = td_path / "repo.zip"
                urllib.request.urlretrieve(archive_url, archive)
                with zipfile.ZipFile(archive) as zf:
                    zf.extractall(td_path / "unzipped")
                roots = [p for p in (td_path / "unzipped").iterdir() if p.is_dir()]
                if len(roots) != 1:
                    raise RuntimeError(f"unexpected zip layout: {roots}")
                shutil.move(str(roots[0]), str(target))
            if _is_leafrust_root(target):
                return target
            raise RuntimeError("zip extracted but train script missing")
        except Exception as exc:
            last_error = exc
            if target.exists():
                shutil.rmtree(target, ignore_errors=True)
    raise RuntimeError(f"ZIP fallback failed: {last_error}")


def _next_free_dir(base: Path) -> Path:
    if not base.exists() or not any(base.iterdir()):
        return base
    if _is_leafrust_root(base):
        return base
    for i in range(1, 50):
        alt = base.parent / (f"{base.name}-src" if i == 1 else f"{base.name}-src{i}")
        if (not alt.exists()) or (alt.is_dir() and not any(alt.iterdir())) or _is_leafrust_root(alt):
            return alt
    return base.parent / f"{base.name}-{os.getpid()}"


def _clone_repo(preferred: Path) -> Path:
    # occupied invalid dir → try update, force wipe, or alternate path
    if preferred.exists() and not _is_leafrust_root(preferred):
        print(f"{preferred} существует, но не LeafRust — пробую git update…")
        if _try_git_update(preferred):
            print("updated in place:", preferred)
            return preferred
        if FORCE_RECLONE:
            print("LEAFRUST_FORCE_RECLONE=1 → удаляю", preferred)
            shutil.rmtree(preferred)
        else:
            preferred = _next_free_dir(preferred)
            print(f"клон в свободный каталог: {preferred}")

    if _is_leafrust_root(preferred):
        return preferred

    preferred.parent.mkdir(parents=True, exist_ok=True)
    git = shutil.which("git")
    if git and (not preferred.exists() or not any(preferred.iterdir())):
        try:
            if preferred.exists() and not any(preferred.iterdir()):
                preferred.rmdir()
            subprocess.run([git, "clone", "--depth", "1", REPO_GIT_URL, str(preferred)], check=True)
            if _is_leafrust_root(preferred):
                return preferred
        except Exception as exc:
            print("git clone не удался, ZIP fallback:", exc)
            if preferred.exists():
                shutil.rmtree(preferred, ignore_errors=True)

    return _download_zip_into(preferred)


REPO = _find_existing_repo() or _clone_repo(DEFAULT_REPO)
SCRIPTS = REPO / "scripts"
sys.path.insert(0, str(SCRIPTS))
os.environ["LEAFRUST_ROOT"] = str(REPO)
try:
    os.chdir(REPO)
except Exception as exc:
    print("chdir warning:", exc)

print("host:", platform.node(), platform.system(), platform.machine())
print("python:", sys.version.split()[0])
print("cwd:", Path.cwd())
print("repo:", REPO)


def _missing_modules(names: list[str]) -> list[str]:
    return [name for name in names if importlib.util.find_spec(name) is None]


def _pip(*args: str) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *args]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


if _missing_modules(["torch", "torchvision"]):
    torch_index = os.environ.get("LEAFRUST_TORCH_INDEX", "https://download.pytorch.org/whl/cu124")
    _pip("torch", "torchvision", "--index-url", torch_index)

requirements = SCRIPTS / "requirements-train-gpu.txt"
required_imports = ["numpy", "matplotlib", "PIL", "huggingface_hub"]

if requirements.exists():
    req_hash = hashlib.sha256(requirements.read_bytes()).hexdigest()
    env_key = hashlib.sha256(
        f"{req_hash}|{sys.executable}|{sys.version_info.major}.{sys.version_info.minor}".encode()
    ).hexdigest()
    marker = Path.home() / ".cache" / "leafrust" / f"requirements-{env_key}.ok"
    if not marker.exists() or _missing_modules(required_imports):
        _pip("-r", str(requirements))
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(req_hash + "\n", encoding="utf-8")

missing = _missing_modules(required_imports)
if missing:
    package_map = {"PIL": "Pillow", "huggingface_hub": "huggingface-hub"}
    _pip(*[package_map.get(x, x) for x in missing])

for mod in ("ensure_train_assets.py", "hf_auth.py"):
    if not (SCRIPTS / mod).exists():
        print("WARNING: missing", SCRIPTS / mod, "— обновите репо или LEAFRUST_FORCE_RECLONE=1")

print("bootstrap OK")


## 2. Импорты, окружение и автоматическая конфигурация

In [ ]:
import csv
import gc
import hashlib
import json
import math
import random
import time
from collections import Counter, defaultdict
from contextlib import nullcontext
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import display, clear_output

from hf_auth import describe_token_source, load_hf_token, login_hf
from ensure_train_assets import ensure_baseline_models, ensure_plantvillage, ensure_torchvision_weights
from train_mobilenet_torch import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    SoftmaxWrapper,
    build_model,
    ensure_background_images,
    export_onnx,
    onnx_to_tflite,
    set_backbone_trainable,
)

# ---------- Пути ----------
DATA_DIR = Path(os.environ.get("LEAFRUST_DATA", str(REPO / "data" / "plantvillage"))).expanduser().resolve()
EXPORT_DIR = Path(os.environ.get("LEAFRUST_EXPORT", str(REPO / "data" / "exports"))).expanduser().resolve()
CHECKPOINT_DIR = Path(os.environ.get("LEAFRUST_CHECKPOINTS", str(REPO / "data" / "checkpoints"))).expanduser().resolve()
OUT_TFLITE = Path(os.environ.get("LEAFRUST_TFLITE", str(EXPORT_DIR / "plantvillage_mobilenet.tflite"))).expanduser().resolve()
BEST_CKPT = CHECKPOINT_DIR / "best_torch.pt"
LAST_CKPT = CHECKPOINT_DIR / "last_torch.pt"
ONNX_PATH = CHECKPOINT_DIR / "leafrust_mobilenet_v3.onnx"
HISTORY_CSV = EXPORT_DIR / "training_history.csv"
METRICS_CSV = EXPORT_DIR / "class_metrics.csv"
REPORT_JSON = EXPORT_DIR / "training_report.json"
CONFUSION_PNG = EXPORT_DIR / "confusion_matrix.png"
PACKAGE_ZIP = EXPORT_DIR / "plantvillage_mobilenet_bundle.zip"

# ---------- Базовые параметры ----------
SEED = int(os.environ.get("LEAFRUST_SEED", "42"))
IMAGE_SIZE = int(os.environ.get("LEAFRUST_IMAGE_SIZE", "224"))
VAL_SPLIT = float(os.environ.get("LEAFRUST_VAL_SPLIT", "0.15"))
MAX_PER_CLASS = int(os.environ["LEAFRUST_MAX_PER_CLASS"]) if os.environ.get("LEAFRUST_MAX_PER_CLASS") else None
VERIFY_IMAGES = os.environ.get("LEAFRUST_VERIFY_IMAGES", "1") != "0"
DOWNLOAD_BASELINE_MODELS = os.environ.get("LEAFRUST_DOWNLOAD_BASELINE", "1") != "0"
SKIP_TFLITE = os.environ.get("LEAFRUST_SKIP_TFLITE", "0") == "1"
AUTO_RESUME = os.environ.get("LEAFRUST_AUTO_RESUME", "1") != "0"

HEAD_MAX_EPOCHS = int(os.environ.get("LEAFRUST_HEAD_EPOCHS", "8"))
FT_MAX_EPOCHS = int(os.environ.get("LEAFRUST_FT_EPOCHS", "30"))
HEAD_PATIENCE = int(os.environ.get("LEAFRUST_HEAD_PATIENCE", "3"))
FT_PATIENCE = int(os.environ.get("LEAFRUST_FT_PATIENCE", "6"))
LR_HEAD = float(os.environ.get("LEAFRUST_LR_HEAD", "0.001"))
LR_FT = float(os.environ.get("LEAFRUST_LR_FT", "0.0001"))
WEIGHT_DECAY = float(os.environ.get("LEAFRUST_WEIGHT_DECAY", "0.0001"))
LABEL_SMOOTHING = float(os.environ.get("LEAFRUST_LABEL_SMOOTHING", "0.05"))
UNFREEZE_BLOCKS = int(os.environ.get("LEAFRUST_UNFREEZE_BLOCKS", "6"))

WORKERS_ENV = os.environ.get("LEAFRUST_WORKERS")
if WORKERS_ENV is not None:
    WORKERS = max(0, int(WORKERS_ENV))
elif platform.system() == "Windows":
    WORKERS = 0
else:
    WORKERS = max(0, min(8, (os.cpu_count() or 2) - 1))

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda" and os.environ.get("LEAFRUST_AMP", "1") != "0"
PIN_MEMORY = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# HF login остаётся необязательным: проект сам решает, нужен ли токен конкретному источнику.
tok = login_hf(load_hf_token())
print("HF token source:", describe_token_source(), "| present:", bool(tok))
print("device:", DEVICE, "| amp:", USE_AMP, "| workers:", WORKERS)
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM={props.total_memory / 2**30:.1f} GiB")

## 3. Датасет, проверка файлов и стратифицированное разбиение

In [ ]:
# Скачиваем только то, чего нет — эти функции уже предусмотрены LeafRust.
ensure_plantvillage(REPO, DATA_DIR, max_per_class=MAX_PER_CLASS)
ensure_torchvision_weights()
ensure_background_images(DATA_DIR)

if DOWNLOAD_BASELINE_MODELS:
    try:
        baselines = ensure_baseline_models(EXPORT_DIR)
        print("baseline models:", {k: str(v) for k, v in baselines.items()})
    except Exception as exc:
        print("WARNING: baseline-модели скачать не удалось; обучение продолжится:", exc)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.82, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12, hue=0.04),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full = datasets.ImageFolder(str(DATA_DIR), transform=train_tf)
full_val = datasets.ImageFolder(str(DATA_DIR), transform=val_tf)
class_names = list(full.classes)


def find_invalid_images(samples: list[tuple[str, int]]) -> list[str]:
    if not VERIFY_IMAGES:
        return []
    bad = []
    print(f"Проверка {len(samples):,} изображений...")
    for i, (path, _) in enumerate(samples, 1):
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad.append(path)
        if i % 5000 == 0:
            print(f"  {i:,}/{len(samples):,}")
    return bad


invalid = find_invalid_images(full.samples)
if invalid:
    bad_set = set(invalid)
    print(f"WARNING: повреждено/нечитаемо: {len(invalid)}. Они исключены из выборки, исходные файлы не удалены.")
    valid_indices = [i for i, (p, _) in enumerate(full.samples) if p not in bad_set]
    full.samples = [full.samples[i] for i in valid_indices]
    full.targets = [full.targets[i] for i in valid_indices]
    full.imgs = full.samples
    full_val.samples = [full_val.samples[i] for i in valid_indices]
    full_val.targets = [full_val.targets[i] for i in valid_indices]
    full_val.imgs = full_val.samples

if len(class_names) < 2:
    raise RuntimeError(f"Нужно минимум 2 класса, найдено: {len(class_names)}")
if len(full) < len(class_names) * 2:
    raise RuntimeError("Слишком мало изображений для надёжного train/validation split")


def stratified_split(targets: list[int], val_fraction: float, seed: int) -> tuple[list[int], list[int]]:
    by_class: dict[int, list[int]] = defaultdict(list)
    for idx, target in enumerate(targets):
        by_class[int(target)].append(idx)
    rng = random.Random(seed)
    train_idx, val_idx = [], []
    for cls, idxs in sorted(by_class.items()):
        rng.shuffle(idxs)
        if len(idxs) == 1:
            n_val = 0
        else:
            n_val = max(1, min(len(idxs) - 1, int(round(len(idxs) * val_fraction))))
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


train_idx, val_idx = stratified_split(full.targets, VAL_SPLIT, SEED)
if not val_idx:
    raise RuntimeError("Validation split получился пустым")

train_counts = Counter(int(full.targets[i]) for i in train_idx)
val_counts = Counter(int(full.targets[i]) for i in val_idx)

print(f"classes={len(class_names)} | total={len(full):,} | train={len(train_idx):,} | val={len(val_idx):,}")
print("min/max train per class:", min(train_counts.values()), max(train_counts.values()))
print("invalid excluded:", len(invalid))

## 4. Автоподбор batch size и DataLoader

In [ ]:
BATCH_ENV = os.environ.get("LEAFRUST_BATCH")


def _amp_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def probe_batch_size(num_classes: int) -> int:
    if BATCH_ENV:
        return max(1, int(BATCH_ENV))
    if DEVICE.type != "cuda":
        return 16 if DEVICE.type == "cpu" else 32

    # Реальный probe: forward + backward + AdamW step на полной trainable модели.
    candidates = [128, 96, 64, 48, 32, 24, 16, 12, 8, 4]
    for bs in candidates:
        m = None
        opt = None
        try:
            torch.cuda.empty_cache()
            m = build_model(num_classes, pretrained=False).to(DEVICE)
            for p in m.parameters():
                p.requires_grad = True
            opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
            x = torch.randn(bs, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            y = torch.randint(0, num_classes, (bs,), device=DEVICE)
            with _amp_context():
                logits = m(x)
                loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            print("auto batch probe OK:", bs)
            return bs
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            print("auto batch probe OOM:", bs)
        finally:
            del m, opt
            gc.collect()
            torch.cuda.empty_cache()
    return 2


BATCH = probe_batch_size(len(class_names))


def make_loaders(batch_size: int) -> tuple[DataLoader, DataLoader]:
    train_loader = DataLoader(
        Subset(full, train_idx),
        batch_size=batch_size,
        shuffle=True,
        num_workers=WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=WORKERS > 0,
        drop_last=False,
    )
    val_loader = DataLoader(
        Subset(full_val, val_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=WORKERS > 0,
        drop_last=False,
    )
    return train_loader, val_loader


train_loader, val_loader = make_loaders(BATCH)
print("batch:", BATCH, "| train steps:", len(train_loader), "| val steps:", len(val_loader))

# Веса классов вычисляются только по train split.
counts = torch.tensor([train_counts.get(i, 0) for i in range(len(class_names))], dtype=torch.float32)
if (counts == 0).any():
    missing = [class_names[i] for i, c in enumerate(counts.tolist()) if c == 0]
    raise RuntimeError(f"В train split отсутствуют классы: {missing}")
class_weights = counts.sum() / (len(counts) * counts)
class_weights = class_weights.to(DEVICE)

# Визуальная sanity-check выборки.
xb, yb = next(iter(train_loader))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
imgs = (xb[:8].cpu() * std + mean).clamp(0, 1).permute(0, 2, 3, 1).numpy()
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax in axes.flat:
    ax.axis("off")
for ax, img, yi in zip(axes.flat, imgs, yb[:8].tolist()):
    ax.imshow(img)
    ax.set_title(class_names[yi][:30], fontsize=8)
plt.tight_layout()
plt.show()

## 5. Надёжное обучение: auto-resume, global-best, scheduler, early stopping

In [ ]:
history = {
    "phase": [], "epoch": [], "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "lr": [], "batch": []
}


def save_history_csv() -> None:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    keys = list(history.keys())
    with HISTORY_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(keys)
        for row in zip(*(history[k] for k in keys)):
            w.writerow(row)


def plot_history() -> None:
    clear_output(wait=True)
    if not history["epoch"]:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    xs = np.arange(1, len(history["epoch"]) + 1)
    axes[0].plot(xs, history["train_loss"], label="train")
    axes[0].plot(xs, history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(xs, history["train_acc"], label="train")
    axes[1].plot(xs, history["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    i = -1
    print(
        f"{history['phase'][i]} epoch={history['epoch'][i]} | "
        f"train loss={history['train_loss'][i]:.4f} acc={history['train_acc'][i]:.4f} | "
        f"val loss={history['val_loss'][i]:.4f} acc={history['val_acc'][i]:.4f} | "
        f"lr={history['lr'][i]:.2e} | batch={history['batch'][i]}"
    )


def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with _amp_context():
                logits = model(xb)
                loss = F.cross_entropy(
                    logits,
                    yb,
                    weight=class_weights if training else None,
                    label_smoothing=LABEL_SMOOTHING if training else 0.0,
                )
            if training:
                if scaler is not None and USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                    optimizer.step()

        bs = yb.size(0)
        total_loss += float(loss.detach()) * bs
        total_correct += int((logits.argmax(dim=1) == yb).sum().item())
        total_seen += bs

    return total_loss / max(1, total_seen), total_correct / max(1, total_seen)


def checkpoint_payload(model, *, global_best, phase_index, epoch_in_phase, optimizer=None, scheduler=None, scaler=None, training_complete=False):
    return {
        "format_version": 2,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "model": model.state_dict(),
        "classes": class_names,
        "global_best_val_acc": float(global_best),
        "val_acc": float(global_best),  # совместимость со старым кодом
        "phase_index": int(phase_index),
        "epoch_in_phase": int(epoch_in_phase),
        "training_complete": bool(training_complete),
        "batch": int(BATCH),
        "image_size": int(IMAGE_SIZE),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "history": history,
    }


def atomic_torch_save(payload, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def compatible_checkpoint(path: Path):
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if ckpt.get("classes") and list(ckpt["classes"]) != class_names:
        print(f"checkpoint {path.name} пропущен: список классов отличается")
        return None
    return ckpt


model = build_model(len(class_names), pretrained=True).to(DEVICE)
global_best = -1.0
resume_phase = 0
resume_epoch = 0
resume_state = None

if AUTO_RESUME:
    resume_state = compatible_checkpoint(LAST_CKPT)

if resume_state is not None:
    model.load_state_dict(resume_state["model"], strict=False)
    global_best = float(resume_state.get("global_best_val_acc", resume_state.get("val_acc", -1.0)))
    resume_phase = int(resume_state.get("phase_index", 0))
    resume_epoch = int(resume_state.get("epoch_in_phase", 0))
    old_history = resume_state.get("history")
    if isinstance(old_history, dict) and set(history).issubset(old_history):
        history = {k: list(old_history[k]) for k in history}
    print(f"AUTO-RESUME: {LAST_CKPT} | phase={resume_phase} epoch={resume_epoch} best={global_best:.4f}")
elif AUTO_RESUME and BEST_CKPT.exists():
    # Миграция со старой версией ноутбука: best_torch.pt используем как warm-start.
    legacy = compatible_checkpoint(BEST_CKPT)
    if legacy is not None:
        model.load_state_dict(legacy["model"], strict=False)
        global_best = float(legacy.get("global_best_val_acc", legacy.get("val_acc", -1.0)))
        resume_phase = 1  # старый best уже как минимум прошёл обучение головы
        resume_epoch = 0
        print(f"LEGACY WARM-START: {BEST_CKPT} | best={global_best:.4f}")

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None

phases = [
    {"name": "head", "epochs": HEAD_MAX_EPOCHS, "patience": HEAD_PATIENCE, "lr": LR_HEAD, "backbone": False},
    {"name": "finetune", "epochs": FT_MAX_EPOCHS, "patience": FT_PATIENCE, "lr": LR_FT, "backbone": True},
]

training_complete = bool(resume_state and resume_state.get("training_complete"))

if training_complete:
    print("Checkpoint отмечен как training_complete=True — повторное обучение не требуется.")
else:
    for phase_index, phase in enumerate(phases):
        if phase_index < resume_phase:
            continue

        set_backbone_trainable(
            model,
            phase["backbone"],
            UNFREEZE_BLOCKS if phase["backbone"] else 0,
        )
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=phase["lr"], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.4, patience=2, min_lr=1e-6
        )

        start_epoch = 1
        bad_epochs = 0
        phase_best = -1.0

        if resume_state is not None and phase_index == resume_phase and resume_epoch > 0:
            start_epoch = resume_epoch + 1
            if resume_state.get("optimizer"):
                try:
                    optimizer.load_state_dict(resume_state["optimizer"])
                except Exception as exc:
                    print("optimizer state не восстановлен, продолжаю с новым optimizer:", exc)
            if resume_state.get("scheduler"):
                try:
                    scheduler.load_state_dict(resume_state["scheduler"])
                except Exception as exc:
                    print("scheduler state не восстановлен:", exc)
            if scaler is not None and resume_state.get("scaler"):
                try:
                    scaler.load_state_dict(resume_state["scaler"])
                except Exception as exc:
                    print("scaler state не восстановлен:", exc)

        print(
            f"=== {phase['name']} | epochs={phase['epochs']} | start={start_epoch} | "
            f"lr={phase['lr']:.2e} | trainable={sum(p.numel() for p in params):,} ==="
        )

        epoch = start_epoch
        while epoch <= phase["epochs"]:
            try:
                tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, scaler)
                va_loss, va_acc = run_epoch(model, val_loader)
            except RuntimeError as exc:
                if DEVICE.type == "cuda" and "out of memory" in str(exc).lower() and BATCH > 2:
                    old_batch = BATCH
                    BATCH = max(2, BATCH // 2)
                    print(f"CUDA OOM на batch={old_batch}; автоматически уменьшаю batch до {BATCH} и повторяю эпоху")
                    optimizer.zero_grad(set_to_none=True)
                    gc.collect()
                    torch.cuda.empty_cache()
                    train_loader, val_loader = make_loaders(BATCH)
                    continue
                raise

            lr_now = float(optimizer.param_groups[0]["lr"])
            history["phase"].append(phase["name"])
            history["epoch"].append(len(history["epoch"]) + 1)
            history["train_loss"].append(float(tr_loss))
            history["train_acc"].append(float(tr_acc))
            history["val_loss"].append(float(va_loss))
            history["val_acc"].append(float(va_acc))
            history["lr"].append(lr_now)
            history["batch"].append(int(BATCH))
            save_history_csv()
            plot_history()

            scheduler.step(va_acc)

            improved_global = va_acc > global_best + 1e-8
            if improved_global:
                global_best = float(va_acc)
                atomic_torch_save(
                    checkpoint_payload(
                        model, global_best=global_best, phase_index=phase_index,
                        epoch_in_phase=epoch, optimizer=optimizer, scheduler=scheduler, scaler=scaler
                    ),
                    BEST_CKPT,
                )
                print(f"GLOBAL BEST saved: {BEST_CKPT} | val_acc={global_best:.4f}")

            if va_acc > phase_best + 1e-5:
                phase_best = float(va_acc)
                bad_epochs = 0
            else:
                bad_epochs += 1

            atomic_torch_save(
                checkpoint_payload(
                    model, global_best=global_best, phase_index=phase_index,
                    epoch_in_phase=epoch, optimizer=optimizer, scheduler=scheduler, scaler=scaler
                ),
                LAST_CKPT,
            )

            if bad_epochs >= phase["patience"]:
                print(f"EARLY STOP {phase['name']}: улучшения нет {bad_epochs} эпох")
                break
            epoch += 1

        # Фаза считается завершённой, даже если завершилась early stopping.
        next_phase = phase_index + 1
        atomic_torch_save(
            checkpoint_payload(
                model, global_best=global_best, phase_index=next_phase,
                epoch_in_phase=0, training_complete=next_phase >= len(phases)
            ),
            LAST_CKPT,
        )
        resume_state = None
        resume_epoch = 0

print("best val_acc:", global_best)
print("best checkpoint:", BEST_CKPT)
print("last checkpoint:", LAST_CKPT)

## 6. Финальная оценка и отчёт по классам

In [ ]:
best = compatible_checkpoint(BEST_CKPT)
if best is None:
    print("WARNING: best_torch.pt недоступен; использую last_torch.pt как аварийный fallback")
    best = compatible_checkpoint(LAST_CKPT)
if best is None:
    raise RuntimeError("Ни best_torch.pt, ни last_torch.pt не найдены или несовместимы с текущим датасетом")
model.load_state_dict(best["model"], strict=True)
model.eval()
global_best = float(best.get("global_best_val_acc", best.get("val_acc", -1.0)))


def detailed_eval(model, loader):
    n = len(class_names)
    confusion = torch.zeros((n, n), dtype=torch.int64)
    total_loss = 0.0
    total_seen = 0
    total_correct = 0
    with torch.inference_mode():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
            yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
            with _amp_context():
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            pred = logits.argmax(1)
            bs = yb.size(0)
            total_loss += float(loss) * bs
            total_seen += bs
            total_correct += int((pred == yb).sum())
            for t, p in zip(yb.cpu().tolist(), pred.cpu().tolist()):
                confusion[t, p] += 1
    return total_loss / total_seen, total_correct / total_seen, confusion


val_loss, val_acc, confusion = detailed_eval(model, val_loader)
rows = []
f1_values = []
for i, name in enumerate(class_names):
    tp = int(confusion[i, i])
    support = int(confusion[i, :].sum())
    predicted = int(confusion[:, i].sum())
    precision = tp / predicted if predicted else 0.0
    recall = tp / support if support else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    f1_values.append(f1)
    rows.append([name, support, precision, recall, f1])

macro_f1 = float(np.mean(f1_values)) if f1_values else 0.0
with METRICS_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["class", "support", "precision", "recall", "f1"])
    w.writerows(rows)

fig_size = max(8, min(20, len(class_names) * 0.45))
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
im = ax.imshow(confusion.numpy())
ax.set_title("Validation confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
if len(class_names) <= 45:
    ax.set_xticks(range(len(class_names)), class_names, rotation=90, fontsize=6)
    ax.set_yticks(range(len(class_names)), class_names, fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
fig.savefig(CONFUSION_PNG, dpi=160, bbox_inches="tight")
plt.show()

print(f"final val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | macro_f1={macro_f1:.4f}")
print("class metrics:", METRICS_CSV)
print("confusion matrix:", CONFUSION_PNG)

## 7. Экспорт ONNX/TFLite + автоматическая проверка

In [ ]:
def timestamped_backup(path: Path) -> Path | None:
    if not path.exists():
        return None
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = path.with_name(f"{path.stem}.{stamp}.bak{path.suffix}")
    shutil.copy2(path, backup)
    return backup


OUT_TFLITE.parent.mkdir(parents=True, exist_ok=True)
backup = timestamped_backup(OUT_TFLITE)
if backup:
    print("backup previous:", backup)

labels_txt = OUT_TFLITE.with_name("labels.txt")
labels_json = OUT_TFLITE.with_suffix(".labels.json")
labels_txt.write_text("\n".join(class_names) + "\n", encoding="utf-8")
labels_json.write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8")

export_onnx(SoftmaxWrapper(model.cpu()), ONNX_PATH, IMAGE_SIZE)
print("ONNX:", ONNX_PATH, f"({ONNX_PATH.stat().st_size / 1e6:.2f} MB)")

# ONNX structural check, если пакет onnx доступен.
onnx_check = {"available": False, "ok": None, "error": None}
try:
    import onnx
    onnx_check["available"] = True
    onnx_model = onnx.load(str(ONNX_PATH))
    onnx.checker.check_model(onnx_model)
    onnx_check["ok"] = True
    print("ONNX checker: OK")
except ImportError:
    print("ONNX checker: пакет onnx не импортирован — пропуск отдельной проверки")
except Exception as exc:
    onnx_check["ok"] = False
    onnx_check["error"] = str(exc)
    raise

if SKIP_TFLITE:
    print("LEAFRUST_SKIP_TFLITE=1 — TFLite conversion skipped")
else:
    onnx_to_tflite(ONNX_PATH, OUT_TFLITE)
    if not OUT_TFLITE.exists() or OUT_TFLITE.stat().st_size == 0:
        raise RuntimeError("TFLite export завершился без корректного файла")
    print("TFLite:", OUT_TFLITE, f"({OUT_TFLITE.stat().st_size / 1e6:.2f} MB)")


def get_tflite_interpreter(path: Path):
    errors = []
    try:
        from ai_edge_litert.interpreter import Interpreter
        return Interpreter(model_path=str(path)), "ai_edge_litert"
    except Exception as exc:
        errors.append(f"ai_edge_litert: {exc}")
    try:
        from tflite_runtime.interpreter import Interpreter
        return Interpreter(model_path=str(path)), "tflite_runtime"
    except Exception as exc:
        errors.append(f"tflite_runtime: {exc}")
    try:
        import tensorflow as tf
        return tf.lite.Interpreter(model_path=str(path)), "tensorflow"
    except Exception as exc:
        errors.append(f"tensorflow: {exc}")
    return None, " | ".join(errors)


def quantize_for_tensor(arr: np.ndarray, detail: dict) -> np.ndarray:
    dtype = detail["dtype"]
    scale, zero = detail.get("quantization", (0.0, 0))
    if np.issubdtype(dtype, np.integer) and scale:
        q = np.round(arr / scale + zero)
        info = np.iinfo(dtype)
        return np.clip(q, info.min, info.max).astype(dtype)
    return arr.astype(dtype)


def dequantize_tensor(arr: np.ndarray, detail: dict) -> np.ndarray:
    scale, zero = detail.get("quantization", (0.0, 0))
    if np.issubdtype(arr.dtype, np.integer) and scale:
        return (arr.astype(np.float32) - zero) * scale
    return arr.astype(np.float32)


tflite_check = {"available": False, "backend": None, "ok": None, "max_abs_diff": None, "same_argmax": None, "error": None}
if not SKIP_TFLITE:
    interpreter, backend = get_tflite_interpreter(OUT_TFLITE)
    if interpreter is None:
        tflite_check["error"] = backend
        print("TFLite runtime не найден — файл создан, runtime smoke-test пропущен")
    else:
        tflite_check["available"] = True
        tflite_check["backend"] = backend
        interpreter.allocate_tensors()
        inp = interpreter.get_input_details()[0]
        out = interpreter.get_output_details()[0]

        sample_x, _ = full_val[val_idx[0]]
        torch_input = sample_x.unsqueeze(0)
        with torch.inference_mode():
            torch_prob = SoftmaxWrapper(model.cpu())(torch_input).numpy()

        arr = torch_input.numpy()
        expected_shape = tuple(int(x) for x in inp["shape"])
        if len(expected_shape) == 4 and expected_shape[-1] == 3 and arr.shape[1] == 3:
            arr = np.transpose(arr, (0, 2, 3, 1))
        arr = quantize_for_tensor(arr, inp)
        interpreter.set_tensor(inp["index"], arr)
        interpreter.invoke()
        tflite_prob = dequantize_tensor(interpreter.get_tensor(out["index"]), out)

        # Приводим к [1, classes].
        tflite_prob = np.asarray(tflite_prob).reshape(1, -1)
        torch_prob = np.asarray(torch_prob).reshape(1, -1)
        if tflite_prob.shape != torch_prob.shape:
            raise RuntimeError(f"TFLite output shape {tflite_prob.shape} != Torch {torch_prob.shape}")
        max_diff = float(np.max(np.abs(tflite_prob - torch_prob)))
        same_argmax = int(np.argmax(tflite_prob)) == int(np.argmax(torch_prob))
        tflite_check.update({"ok": True, "max_abs_diff": max_diff, "same_argmax": bool(same_argmax)})
        print(f"TFLite smoke-test ({backend}): OK | same_argmax={same_argmax} | max_abs_diff={max_diff:.6g}")

## 8. Android assets, итоговый отчёт и ZIP-пакет

In [ ]:
# Автодоставка в Android-проект: только если каталог действительно существует,
# либо если путь явно задан через LEAFRUST_ANDROID_MODELS.
android_env = os.environ.get("LEAFRUST_ANDROID_MODELS")
android_models = Path(android_env).expanduser().resolve() if android_env else (
    REPO / "android" / "app" / "src" / "main" / "assets" / "models"
)
android_copied = []
if not SKIP_TFLITE and (android_models.exists() or android_env):
    android_models.mkdir(parents=True, exist_ok=True)
    for src in [OUT_TFLITE, labels_txt, labels_json]:
        dst = android_models / src.name
        shutil.copy2(src, dst)
        android_copied.append(str(dst))
    print("Android assets updated:", android_models)
else:
    print("Android assets: каталог не найден — копирование не требуется")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


artifacts = [BEST_CKPT, LAST_CKPT, ONNX_PATH, labels_txt, labels_json, HISTORY_CSV, METRICS_CSV, CONFUSION_PNG]
if not SKIP_TFLITE:
    artifacts.append(OUT_TFLITE)
artifact_info = {}
for p in artifacts:
    if p.exists():
        artifact_info[p.name] = {
            "path": str(p),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        }

report = {
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "repo": str(REPO),
    "data_dir": str(DATA_DIR),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "classes": class_names,
    "num_classes": len(class_names),
    "total_images": len(full),
    "train_images": len(train_idx),
    "val_images": len(val_idx),
    "invalid_images_excluded": invalid,
    "image_size": IMAGE_SIZE,
    "batch_final": BATCH,
    "workers": WORKERS,
    "amp": USE_AMP,
    "best_val_acc": global_best,
    "final_val_acc": float(val_acc),
    "final_val_loss": float(val_loss),
    "macro_f1": macro_f1,
    "onnx_check": onnx_check,
    "tflite_check": tflite_check,
    "android_copied": android_copied,
    "artifacts": artifact_info,
}
REPORT_JSON.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

# Собираем переносимый bundle без checkpoint'ов: для приложения/релиза нужны модель, labels и отчёт.
package_files = [ONNX_PATH, labels_txt, labels_json, HISTORY_CSV, METRICS_CSV, CONFUSION_PNG, REPORT_JSON]
if not SKIP_TFLITE:
    package_files.insert(0, OUT_TFLITE)
with zipfile.ZipFile(PACKAGE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in package_files:
        if p.exists():
            zf.write(p, arcname=p.name)

print("\n=== PIPELINE COMPLETE ===")
print(f"val_acc={val_acc:.4f} | macro_f1={macro_f1:.4f}")
print("report:", REPORT_JSON)
print("bundle:", PACKAGE_ZIP, f"({PACKAGE_ZIP.stat().st_size / 1e6:.2f} MB)")
print("TFLite:", OUT_TFLITE if not SKIP_TFLITE else "skipped")

## Результат

После успешного **Run All** основными выходными файлами будут:

- `data/exports/plantvillage_mobilenet.tflite` — модель для приложения;
- `data/exports/labels.txt` и `.labels.json` — классы;
- `data/exports/training_history.csv` — история эпох;
- `data/exports/class_metrics.csv` — precision/recall/F1;
- `data/exports/confusion_matrix.png` — confusion matrix;
- `data/exports/training_report.json` — параметры запуска, метрики, проверки и SHA-256 артефактов;
- `data/exports/plantvillage_mobilenet_bundle.zip` — готовый пакет для переноса/релиза;
- `data/checkpoints/last_torch.pt` — автоматическое продолжение после обрыва;
- `data/checkpoints/best_torch.pt` — лучший checkpoint по validation accuracy.

Для повторного запуска ничего переключать не нужно: ноутбук сам продолжит незавершённое обучение либо, если `training_complete=True`, сразу перейдёт к оценке и экспорту.